# exp140 z_slope_posthoc_correction_on_pfbeam_candidates train

Train-side audit for gated `dZ/dMD` posthoc correction on fixed PF/Beam candidates.

## Contents

1. Setup and configuration
2. Input and audit contract
3. Run Z-slope posthoc correction audit
4. Preview outputs
5. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from z_slope_posthoc_correction_on_pfbeam_candidates import run_audit

config = load_config()
paths = ExperimentPaths()
paths.ensure_output_dirs()

print(json.dumps({
    'experiment': get_nested(config, 'experiment.name'),
    'route': get_nested(config, 'experiment.route'),
    'status': get_nested(config, 'experiment.status'),
    'parent': get_nested(config, 'lineage.parent'),
    'cache_parent': get_nested(config, 'lineage.cache_parent'),
    'mode': get_nested(config, 'audit.mode'),
}, indent=2, sort_keys=True))

## 2. Input and audit contract

In [ ]:
print(json.dumps({
    'feature_cache': get_nested(config, 'data.exp072_train_feature_cache_local'),
    'kernel_sources': get_nested(config, 'runtime.kaggle.kernel_sources'),
    'primary_baseline': get_nested(config, 'audit.primary_baseline'),
    'representative_wells': get_nested(config, 'audit.representative_wells'),
    'z_slope': get_nested(config, 'model.z_slope'),
}, indent=2, sort_keys=True))

## 3. Run Z-slope posthoc correction audit

In [ ]:
summary = run_audit(config=config, paths=paths)
print(json.dumps({
    'rows': summary['rows'],
    'wells': summary['wells'],
    'variant_count': summary['variant_count'],
    'primary_baseline': summary['primary_baseline'],
    'best_z_slope_variant': summary['best_z_slope_variant'],
}, indent=2, sort_keys=True))

## 4. Preview outputs

In [ ]:
artifact_dir = paths.artifacts_dir
metrics_path = artifact_dir / 'exp140_z_slope_posthoc_correction_on_pfbeam_candidates_candidate_metrics.csv'
bucket_path = artifact_dir / 'exp140_z_slope_posthoc_correction_on_pfbeam_candidates_bucket_metrics.csv'
by_well_path = artifact_dir / 'exp140_z_slope_posthoc_correction_on_pfbeam_candidates_by_well.csv'
group_path = artifact_dir / 'exp140_z_slope_posthoc_correction_on_pfbeam_candidates_group_metrics.csv'

for path in [metrics_path, bucket_path, by_well_path, group_path]:
    print(path, path.exists(), path.stat().st_size if path.exists() else None)

metrics = pd.read_csv(metrics_path)
display(metrics.head(20))

## 5. Metrics and artifacts

In [ ]:
group_metrics = pd.read_csv(group_path)
display(group_metrics[group_metrics['group'].isin(['all', 'near_000_050', 'longtail_1000_plus', 'z_abs_top_quartile'])].head(40))

print('summary:', summary['artifacts']['summary'])
print('sha:', json.dumps(summary['artifact_sha256'], indent=2, sort_keys=True))